# Qwen2.5-0.5B Batch Inference on Ray Data - Colab T4 Demo

Companion notebook to the KubeRay batch inference submission.

**What this is:** the same `QwenPredictor` Ray Data UDF that runs inside
the production RayCluster (`inference/jobs/batch_infer.py`), executed on
Colab's free T4 GPU. The point is to show the inference core is identical
across environments - the kind cluster proves the KubeRay integration,
this notebook proves the pipeline on real GPU hardware.

**What this is NOT:** a replacement for the KubeRay submission. Colab
has no Kubernetes, no RayCluster CRD, no PVC, no FastAPI control plane.
Single-VM Ray only.

## Before you run

1. `Runtime → Change runtime type → T4 GPU`
2. Run cells top-to-bottom. First run takes ~3 min (model download + Ray start).

## 1. Install dependencies

Pin the same Ray version as the production image (`ray==2.54.1`). Colab
ships recent torch + transformers, so we only force Ray + accelerate.

In [ ]:
!pip install -q "ray[data]==2.54.1" "transformers>=4.44,<5" "accelerate>=0.33" "python-ulid>=3.0,<4"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Start Ray (single-node, local)

Colab is one VM, so this is a local Ray instance - no head/worker split.
The UDF code below is byte-identical to what runs on the multi-pod
RayCluster in `k8s/ray/raycluster.yaml`; only the cluster topology differs.

In [ ]:
import ray
ray.shutdown()
ray.init(log_to_driver=True, include_dashboard=False)
print(ray.cluster_resources())

## 3. `QwenPredictor` - the production UDF

Lifted from `inference/jobs/batch_infer.py` with two small changes for
the T4:

- `torch_dtype=torch.float16` + `device_map="cuda"` (prod uses bf16 on CPU)
- `batch_size=16` instead of 8 (more VRAM headroom than kind workers)

Everything else - chat template, prompt-token slicing, finish-reason
heuristic, per-row error isolation - is the same.

In [ ]:
import time
from typing import Any


class QwenPredictor:
    """Stateful Ray Data actor: loads Qwen once, serves generations per batch."""

    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B-Instruct", max_tokens: int = 128) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self._torch = torch
        self._max_tokens = max_tokens
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.bfloat16

        t0 = time.monotonic()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).to(device)
        self.model.eval()
        self._device = device
        print(f"Loaded {model_name} in {time.monotonic() - t0:.1f}s on {device} ({dtype})")

    def _format_prompt(self, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_prompt},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def __call__(self, batch: dict[str, Any]) -> dict[str, Any]:
        torch = self._torch
        ids = [str(v) for v in batch["id"]]
        prompts = [str(v) for v in batch["prompt"]]
        n = len(prompts)

        responses: list[str | None] = [None] * n
        finish_reasons: list[str | None] = [None] * n
        prompt_tokens_out = [0] * n
        completion_tokens_out = [0] * n
        errors: list[str | None] = [None] * n

        for idx, prompt in enumerate(prompts):
            try:
                text = self._format_prompt(prompt)
                inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(self._device)
                prompt_tokens_out[idx] = int(inputs["input_ids"].shape[1])
                with torch.no_grad():
                    output = self.model.generate(
                        **inputs,
                        max_new_tokens=self._max_tokens,
                        do_sample=False,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                    )
                gen = output[0][inputs["input_ids"].shape[1]:]
                completion_tokens_out[idx] = int(gen.shape[0])
                responses[idx] = self.tokenizer.decode(gen, skip_special_tokens=True).strip()
                last = int(gen[-1]) if len(gen) else -1
                finish_reasons[idx] = "stop" if last == self.tokenizer.eos_token_id else "length"
            except Exception as exc:
                errors[idx] = f"{type(exc).__name__}: {exc}"

        return {
            "id": ids,
            "prompt": prompts,
            "response": responses,
            "finish_reason": finish_reasons,
            "prompt_tokens": prompt_tokens_out,
            "completion_tokens": completion_tokens_out,
            "error": errors,
        }

## 4. Build a 50-prompt dataset

Same shape as `/data/batches/<id>/input.jsonl` in prod: one row per
prompt with `id` (ULID) and `prompt` columns.

In [ ]:
from ulid import ULID

PROMPTS = [
    "What is 2 + 2?",
    "Write a haiku about distributed systems.",
    "Explain RAG in one sentence.",
    "Name three benefits of Kubernetes.",
    "What does KubeRay do?",
    "Write a Python one-liner to reverse a string.",
    "Who wrote the Iliad?",
    "Summarize the CAP theorem.",
    "What is the capital of Uganda?",
    "Give a concise definition of idempotence.",
] * 5  # 50 prompts

rows = [{"id": str(ULID()), "prompt": p} for p in PROMPTS]
ds = ray.data.from_items(rows)
print(f"Dataset: {ds.count()} rows, schema = {ds.schema()}")

## 5. Run `map_batches` and time it

Single GPU → `concurrency=1`. On a multi-worker RayCluster this is
`concurrency=N_workers` with the same UDF.

In [ ]:
t0 = time.monotonic()
out_ds = ds.map_batches(
    QwenPredictor,
    fn_constructor_kwargs={"model_name": "Qwen/Qwen2.5-0.5B-Instruct", "max_tokens": 128},
    concurrency=1,
    batch_size=16,
    num_gpus=1,
)
results = out_ds.take_all()
elapsed = time.monotonic() - t0

total_completion = sum(r["completion_tokens"] for r in results)
failed = sum(1 for r in results if r["error"])
print(f"{len(results)} prompts in {elapsed:.1f}s - {total_completion} tokens generated")
print(f"Throughput: {total_completion / elapsed:.1f} tok/s, {len(results) / elapsed:.2f} prompts/s")
print(f"Failures: {failed}")

## 6. Inspect a few completions

In [ ]:
import json
for r in results[:5]:
    print(json.dumps({"prompt": r["prompt"], "response": r["response"], "finish": r["finish_reason"]}, indent=2))
    print("---")

## 7. Write results.jsonl (same contract as prod)

In the real system this is what the FastAPI control plane streams back
from the PVC when the client calls `GET /v1/batches/{id}/results`.

In [ ]:
from pathlib import Path
out = Path("/content/results.jsonl")
with out.open("w") as fh:
    for r in results:
        fh.write(json.dumps(r) + "\n")
print(f"Wrote {out} ({out.stat().st_size} bytes)")

## 8. Teardown

In [ ]:
ray.shutdown()

## What to take to the debrief

Two numbers from §5 to cite alongside the kind CPU benchmark in
`docs/TECHNICAL_REPORT.md §3.5.1`:

- **Tok/s on T4** vs. tok/s on kind CPU worker → order-of-magnitude
  evidence that the bottleneck is the CPU backend, not the pipeline.
- **Prompts/s** → direct input for capacity-planning slide.

The one-line pitch:

> "The UDF is the same in both places. kind proves the KubeRay
> integration; Colab proves the GPU path. Swapping to a GPU RayCluster
> in prod is a node-pool change, not a code change."
